In [ ]:
import numpy as np
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.windows import from_bounds
from pprint import pprint
from affine import Affine

In [ ]:
TARGET_RES = 0.1  # 10 cm

In [68]:

def is_geographic(crs):
    return crs.is_geographic


def choose_target_crs(src1, src2):
    crs1, crs2 = src1.crs, src2.crs

    if is_geographic(crs1) and is_geographic(crs2):
        return "EPSG:3857"

    if not is_geographic(crs1) and is_geographic(crs2):
        return crs1

    if is_geographic(crs1) and not is_geographic(crs2):
        return crs2

    # both projected but maybe different
    return crs1


def reproject_to_target(src, target_crs, bounds):
    transform, width, height = calculate_default_transform(
        src.crs,
        target_crs,
        src.width,
        src.height,
        *bounds,
        resolution=TARGET_RES
    )

    band_count = min(3, src.count)
    bands = list(range(1, band_count + 1))

    data = np.zeros((band_count, height, width), dtype=np.uint8)

    for i, b in enumerate(bands):
        reproject(
            source=rasterio.band(src, b),
            destination=data[i],
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=transform,
            dst_crs=target_crs,
            resampling=Resampling.nearest,
        )

    return data, transform


def write_data(filaneme:str, data:np.ndarray, transform:Affine, crs:rasterio.crs.CRS):
    with rasterio.open(filaneme, mode='w', **{
        "count" : data.shape[0],
        "height" : data.shape[1],
        "width" : data.shape[2],
        "transform" : transform,
        "nodata" : 0,
        "crs" : crs,
        "dtype" : data.dtype
    }) as dst:
        dst.write(data) 

def process(tif1_path, tif2_path):
    with rasterio.open(tif1_path) as src1, rasterio.open(tif2_path) as src2:

        print("=== INPUT CRS ===")
        print(src1.crs, src2.crs)

        # 1. Choose CRS
        target_crs = choose_target_crs(src1, src2)
        print("\n=== TARGET CRS ===")
        print(target_crs)

        # 2. Compute intersection in original CRS (approx safe)
        b1, b2 = src1.bounds, src2.bounds

        left   = max(b1.left, b2.left)
        right  = min(b1.right, b2.right)
        bottom = max(b1.bottom, b2.bottom)
        top    = min(b1.top, b2.top)

        if left >= right or bottom >= top:
            raise ValueError("No overlap!")

        common_bounds = (left, bottom, right, top)

        # 3. Reproject both to target CRS + 10cm
        data1, transform1 = reproject_to_target(src1, target_crs, common_bounds)
        data2, transform2 = reproject_to_target(src2, target_crs, common_bounds)

        # 4. Ensure same shape (important!)
        min_h = min(data1.shape[1], data2.shape[1])
        min_w = min(data1.shape[2], data2.shape[2])



        data1 = data1[:, :min_h, :min_w]
        data2 = data2[:, :min_h, :min_w]


        print("\n=== OUTPUT ===")
        print("data1 Shape:", data1.shape)
        print("data2 Shape:", data2.shape)
        print("data1 transform:", transform1)
        print("data2 transform:", transform2)

        return data1, data2, transform1, transform2, target_crs

In [57]:
data1, data2, transform1, transform2, target_crs = process('Phase1.tif', 'Phase2.tif')

=== INPUT CRS ===
PROJCS["WGS 84 / UTM zone 45N",GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",87],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]] PROJCS["WGS 84 / UTM zone 45N",GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",87],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]

=

In [69]:
write_data('Phase1_processed.tif', data1, transform1, target_crs)
write_data('Phase2_processed.tif', data2, transform2, target_crs)